# Dubby 🐨 on Google Colab

**YouTube dubbing studio · English → Egyptian Arabic (مصري) / Modern Standard Arabic (فصحى)**

Run the cells **one by one, top to bottom**. At the end you get a link to the full Dubby web studio running on Colab's GPU, with nothing installed on your computer.

| Step | What happens |
|---|---|
| 1️⃣ | Check the GPU |
| 2️⃣ | Get the Dubby code |
| 3️⃣ | Install Node.js 22 (UI build + YouTube JS challenges) |
| 4️⃣ | Install the **core** engines: WhisperX, Cohere Transcribe / CohereX, Metro-ASR, oddadmix translators, VoiceTut & Lahgtna OmniVoice |
| 5️⃣ | *(optional)* Qwen3-ASR + QwenCleo-ASR in their own venv, NVIDIA Parakeet, Demucs |
| 6️⃣ | Hugging Face token (needed for the gated Cohere models) |
| 7️⃣ | Build the UI and launch the studio |

> 💡 Use **Runtime → Change runtime type → T4 GPU** (or better) before starting.

## 1️⃣ Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "⚠️ No GPU — switch the runtime to a GPU for usable speed"
!python --version

## 2️⃣ Get the Dubby code

In [ ]:
#@title Clone the repository { display-mode: "form" }
REPO_URL = "https://github.com/MohammedAly22/dubby"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
import os, subprocess
if os.path.isdir("/content/dubby/.git"):
    subprocess.run(["git", "-C", "/content/dubby", "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "/content/dubby"], check=True)
%cd /content/dubby
!ls

## 3️⃣ Install Node.js 22

In [ ]:
#@title Node.js (used to build the UI and by yt-dlp for YouTube) { display-mode: "form" }
NODE_VERSION = "22.20.0"  #@param {type:"string"}
import os, subprocess
node_dir = f"/usr/local/lib/node-v{NODE_VERSION}-linux-x64"
if not os.path.isdir(node_dir):
    subprocess.run(f"curl -fsSL https://nodejs.org/dist/v{NODE_VERSION}/node-v{NODE_VERSION}-linux-x64.tar.xz | tar -xJ -C /usr/local/lib", shell=True, check=True)
os.environ["PATH"] = f"{node_dir}/bin:" + os.environ["PATH"]
!node --version && npm --version
!apt-get -qq install -y ffmpeg > /dev/null && ffmpeg -version | head -n 1

## 4️⃣ Install the core engines (≈ 5 min)

In [ ]:
#@title Core install { display-mode: "form" }
INSTALL_DEMUCS = True  #@param {type:"boolean"}
!pip install -q torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -e "/content/dubby[core]"
if INSTALL_DEMUCS:
    !pip install -q demucs
print("✅ core engines installed")

## 5️⃣ Optional engine families

* **Qwen3-ASR / QwenCleo-ASR** need `transformers 4.57`, so Dubby runs them in a separate venv at `/content/envs/dubby-qwen`, which it detects automatically.
* **NVIDIA Parakeet** needs NeMo (`/content/envs/dubby-nemo`).

In [ ]:
#@title Qwen & NeMo families { display-mode: "form" }
INSTALL_QWEN = True  #@param {type:"boolean"}
INSTALL_NEMO = False  #@param {type:"boolean"}
!pip install -q uv
if INSTALL_QWEN:
    !uv venv -q --python 3.12 /content/envs/dubby-qwen
    !uv pip install -q --python /content/envs/dubby-qwen/bin/python torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
    !uv pip install -q --python /content/envs/dubby-qwen/bin/python -e "/content/dubby[qwen]"
    !uv pip install -q --python /content/envs/dubby-qwen/bin/python qwencleo-asr --no-deps
    print("✅ qwen family ready")
if INSTALL_NEMO:
    !uv venv -q --python 3.12 /content/envs/dubby-nemo
    !uv pip install -q --python /content/envs/dubby-nemo/bin/python torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
    !uv pip install -q --python /content/envs/dubby-nemo/bin/python -e "/content/dubby[nemo]"
    print("✅ nemo family ready")

## 6️⃣ Hugging Face token

Cohere Transcribe (and CohereX) are **gated**: accept the terms on
[cohere-transcribe-03-2026](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026) and
[cohere-transcribe-arabic-07-2026](https://huggingface.co/CohereLabs/cohere-transcribe-arabic-07-2026), then paste a *read* token.
Tip: store it as a Colab secret named `HF_TOKEN`.

In [ ]:
#@title Token { display-mode: "form" }
HF_TOKEN = ""  #@param {type:"string"}
import os
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ logged in to Hugging Face")
else:
    print("ℹ️ No token: gated Cohere engines will be unavailable, everything else works.")

## 7️⃣ Build the UI & check engines

In [ ]:
!cd /content/dubby && dubby build-ui
!FORCE_COLOR=1 COLUMNS=110 dubby doctor

## 🚀 Launch the studio

The terminal output (stage banners, live transcripts, translations, clip timings) goes to `/content/dubby.log`; the next cell shows it live.
Pick **Colab proxy** (quickest) or **Cloudflare tunnel** (a public `trycloudflare.com` link that also works in another tab or on your phone).

In [ ]:
#@title Start Dubby { display-mode: "form" }
TUNNEL = "colab-proxy"  #@param ["colab-proxy", "cloudflare"]
EXPORT_TO_DRIVE = False  #@param {type:"boolean"}
import os, subprocess, time, json, urllib.request, re

os.environ.update({"DUBBY_HOME": "/content/Dubby", "FORCE_COLOR": "1", "COLUMNS": "120", "PYTHONUNBUFFERED": "1"})
if EXPORT_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/Dubby", exist_ok=True)
    json.dump({"export_dir": "/content/drive/MyDrive/Dubby"}, open("/content/Dubby/settings.json", "w"))

log = open("/content/dubby.log", "w")
server = subprocess.Popen(["dubby", "serve", "--host", "0.0.0.0", "--port", "8765", "--no-open"], stdout=log, stderr=subprocess.STDOUT, cwd="/content/dubby")
for _ in range(90):
    try:
        urllib.request.urlopen("http://127.0.0.1:8765/api/settings", timeout=2)
        break
    except Exception:
        time.sleep(1)
print("✅ Dubby is running (pid", server.pid, ")")

if TUNNEL == "cloudflare":
    if not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared", shell=True, check=True)
    cf = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8765"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in cf.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            print("🐨 Open the studio →", m.group(0))
            break
else:
    from google.colab import output
    from google.colab.output import eval_js
    print("🐨 Open the studio →", eval_js("google.colab.kernel.proxyPort(8765)"))
    output.serve_kernel_port_as_window(8765, anchor_text="Open Dubby studio in a new tab")

### 📟 Live terminal (re-run any time)

In [ ]:
#@title Show the last studio log lines { display-mode: "form" }
LINES = 80  #@param {type:"integer"}
!tail -n {LINES} /content/dubby.log

## 🤖 Optional: dub headlessly from the notebook

The same pipeline without the UI; the project also shows up in the studio.

In [ ]:
#@title Headless dub { display-mode: "form" }
URL = "https://www.youtube.com/watch?v=jNQXAC9IVRw"  #@param {type:"string"}
TARGET = "arz"  #@param ["arz", "arb"]
ASR = "whisperx"  #@param ["whisperx", "cohere-transcribe", "coherex", "qwen3-asr", "parakeet"]
TRANSLATION = "emhotob"  #@param ["emhotob", "jisr", "masrawy", "llm"]
TTS = "voicetut"  #@param ["voicetut", "lahgtna-omnivoice"]
VOICE = "auto"  #@param ["auto", "preset:Mohamed", "preset:Asmaa", "preset:Sayed"]
!FORCE_COLOR=1 COLUMNS=120 DUBBY_HOME=/content/Dubby dubby dub "{URL}" --target {TARGET} --asr {ASR} --translation {TRANSLATION} --tts {TTS} --voice {VOICE} --export /content/exports

## 🛑 Stop the studio

In [ ]:
try:
    server.terminate()
    print("Stopped.")
except NameError:
    print("Server was not started from this notebook session.")